# 19 — The De Palma limitation: when quantum optimal transport bounds the algorithm

Turn the geometry around. Across this module you used quantum optimal transport to *measure* — to read the coupling between two systems off a density matrix. Here you meet a result where quantum optimal transport becomes a *proof technique*: transport inequalities, built on Wasserstein distances of order 1, limit what shallow variational quantum algorithms can achieve, drawing a provable line around a whole family of near-term quantum methods.

**Prerequisites:** `03/08` (Kantorovich duality, the Lipschitz side of the Wasserstein distance), `04/03` and `04/09` (the families of quantum transport), and `04/16` (the conclusion of the capstone).

**What you'll be able to do**
- Explain the turn from QOT-as-*measurement* to QOT-as-*proof-technique*: optimal transport, which read coupling, now bounds a computation, from a different corner of the field.
- State the **De Palma, Marvian, Rouzé & Stilck França (2023)** limitation theorem *accurately* — a limitation on *shallow, local* variational circuits (VQE / QAOA), conditioned on depth and locality, not a blanket claim that "VQE does not work".
- Give the mechanism in one line: a shallow circuit has a small light cone, so its output obeys a transport (Poincaré) inequality and every Lipschitz quantity concentrates, as on a product state.
- Situate the module's coupling SDP inside the **broader QOT taxonomy** — the quantum-$W_1$ (Lipschitz / observable) family is a *different corner* from the coupling object you built, and it is the corner this theorem lives in.
- Name three honest **open problems** at this frontier.

You arrive having closed the applied arc. In `04/16` you wrote the capstone's accounting from measurements: a richer-embedding quantum measure reads past the phase-locking value, as far as a classical statistic of the same harmonics and no further. That was QOT pointed *at data*, as an instrument. This notebook points optimal transport the other way: at *computation itself*. It asks not "what does this state's coupling look like?" but "how far can a shallow quantum circuit even travel?" — and answers it with a transport bound. One small computation makes the mechanism visible. This is mostly a result to understand, state precisely, and place on the map.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS

np.random.seed(42)
viz.use_course_style()


## 1. From measuring coupling to bounding algorithms

Every use of quantum optimal transport in this module has been *constructive*. You built the coupling SDP (`04/04`) to turn two density matrices into a transport cost; you added an entropy bonus (`04/06`–`04/08`) to make it scale; you embedded signals into states (`04/10`) and read the coupling between two oscillators off the resulting joint density (`04/11`–`04/16`). In every case QOT was an **instrument**: feed it states, read out a number that means *how far apart* or *how coupled*.

There is a second way to use a geometry, and it is the subject of this notebook. A distance does not only *report* how far two objects are — it *constrains* how far you can get from a starting point in a bounded number of moves. If each step of a process can increase a distance by at most some amount, then after a few steps you are still provably close to where you began. That is a statement about *reachability*, and it is exactly the kind of statement that produces a **limitation theorem**: if the good answers all live far away in the distance, and your process cannot travel far, then your process cannot reach the good answers.

This is the turn De Palma, Marvian, Rouzé & Stilck França (2023) make. The geometry is that of **Wasserstein distances of order 1**, on bit strings and on qubits, a corner of quantum optimal transport distinct from the coupling object this module built (we place the two in §3). The "process" is a **variational quantum algorithm**: a parametrised circuit $U(\theta)$ applied to a simple reference state, tuned to minimise the energy $\langle \psi(\theta) | H | \psi(\theta) \rangle$ of some cost Hamiltonian $H$. The limitation is that the output of a *shallow, local* circuit concentrates almost as a product state does, and on hard instances a good answer needs an output that does not. Optimal transport, which this module used to *measure* coupling, is here the instrument that *proves a boundary* on quantum computation.

Hold the shift in one sentence: **optimal transport is constructive when you point it at data and restrictive when you point it at an algorithm**, and the two uses rely on different objects of the same family.


## 2. The De Palma et al. theorem

The value of this result is in its precision: overstated, it becomes a myth; stated as the paper states it, it is a genuine boundary.

### The object: a variational quantum algorithm

A **variational quantum algorithm** (VQA) — the family that includes the *variational quantum eigensolver* (VQE) and the *quantum approximate optimisation algorithm* (QAOA) — prepares a trial state by applying a parametrised circuit to a fixed, simple reference:

$$ |\psi(\theta)\rangle \;=\; U(\theta)\,|\psi_0\rangle, \qquad U(\theta) = U_L(\theta_L)\cdots U_1(\theta_1), $$

where $|\psi_0\rangle$ is typically a **product state** such as $|0\cdots 0\rangle$, and each layer $U_\ell$ is built from **local** gates that touch a few neighbouring qubits. The algorithm tunes $\theta$ to minimise the energy $\mathrm{tr}\!\big(H\,|\psi(\theta)\rangle\langle\psi(\theta)|\big)$ of a cost Hamiltonian $H$; for QAOA, $H$ encodes a combinatorial problem such as Max-Cut. Two numbers describe the circuit's reach: its **depth** $L$ and its **light cone**, the largest number of output qubits that one input qubit can influence. With local gates the light cone grows with depth, at most exponentially.

### The tool: a transport inequality on the output

Measure the output in the computational basis and you get a probability distribution $\mu$ over strings of $n$ bits. The natural distance between two strings is the **Hamming distance**, the number of bits in which they differ, and a function $f$ of strings is *Lipschitz* with constant $\|f\|_L$ if one bit flip changes it by at most $\|f\|_L$. The Wasserstein distance of order 1 between two such distributions is the largest gap $\mathbb{E}_\nu[f] - \mathbb{E}_\mu[f]$ over functions with $\|f\|_L \le 1$: the Kantorovich duality of `03/08`, with the Hamming distance as cost.

The paper's main tool for shallow circuits is a **(2, ∞)-Poincaré inequality**, due to Milman: the variance of every Lipschitz function is at most $n$ times a constant,
$$ \mathrm{Var}_\mu(f) \;\le\; C\, n\, \|f\|_L^2 . $$
The paper proves a quantum version for the output state (its Proposition III.2, in the arXiv version): for a noiseless circuit started in a product state, the constant $C$ is of the order of the square of the light cone. A product state satisfies it with $C = 1$.

### The consequence: concentration

The inequality has direct consequences (Theorem III.1 of the arXiv version). Every Lipschitz observable **concentrates**: it is unlikely to fall more than a few $\sqrt{C n}$ from its mean. And two sets of output strings that each carry a fair share of the probability can be only about $\sqrt{C n}$ apart in Hamming distance. A GHZ state violates this: its two strings $0\cdots0$ and $1\cdots1$ have probability $\tfrac12$ each and are $n$ apart. For Max-Cut on some regular bipartite graphs of degree $D$, the paper turns this into a depth requirement: QAOA needs depth at least $\log(n/576) / (2\log(D+1))$ to outperform efficient classical algorithms on some instances.

### State it accurately — and state what it does *not* say

> **De Palma, Marvian, Rouzé & Stilck França (2023).** Using transport and concentration inequalities from quantum optimal transport, they prove limitations for *shallow* variational circuits. In the **noiseless** regime, the output of a circuit with a small light cone obeys a Poincaré inequality, so Lipschitz observables concentrate as they do on a product state, which forces QAOA to logarithmic depth before it can outperform efficient classical algorithms on some Max-Cut instances. In the **noisy** regime (local depolarising noise at rate $p$), they prove that at depths $L = \mathcal{O}(1/p)$ it is exponentially unlikely that the circuit's outcome outperforms efficient *classical* algorithms on problems such as Max-Cut.

The honest reading carries three guardrails:

- **It is conditional on depth and locality.** The limitation bites when the light cone is small, which is the near-term (NISQ) regime the paper targets. A circuit deep enough for its light cone to cover the system escapes it. The depth is the dial.
- **It is not "VQE does not work".** It is a no-go for a *regime* and a class of problems. It says that claims of an advantage from a shallow circuit on such problems cannot hold, not that variational methods are useless everywhere.
- **It concerns Lipschitz quantities.** Concentration is stated for observables whose value changes by a bounded amount when one qubit changes, such as few-body or geometrically local cost Hamiltonians like Max-Cut.

That precision is the result: a shallow circuit's output concentrates the way a product state's does, and a good answer to a hard instance needs an output that does not.

A small computation makes the mechanism visible. Take the **Hamming weight** $w$ of the measured string, its number of 1s: one bit flip changes $w$ by at most 1, so $\|w\|_L = 1$, and the Poincaré inequality says $\mathrm{Var}(w) \le C n$. Watch $\mathrm{Var}(w)/n$ as the number of qubits grows.

**By hand first.** A cat state on $k$ qubits, $(|0\cdots0\rangle + |1\cdots1\rangle)/\sqrt2$, gives $w = 0$ or $w = k$ with probability $\tfrac12$ each, so $\mathrm{Var}(w) = k^2/4$. A ladder of CNOT gates of depth $L$, after a Hadamard on the first qubit, builds a cat on the first $L+1$ qubits and leaves the others at $|0\rangle$. So $\mathrm{Var}(w) = (L+1)^2/4$ whatever $n$ is: at fixed depth, $\mathrm{Var}(w)/n$ falls as $n$ grows. The GHZ state is the cat on all $n$ qubits; it needs depth $n-1$ and gives $\mathrm{Var}(w)/n = n/4$, which grows without bound, so no fixed constant $C$ fits it. The cell below checks these values and adds circuits of depth 3 made of random two-qubit gates.

In [ ]:
# A computed toy of the mechanism: Var(w)/n for the Hamming weight w of the measured string.
H_GATE = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], dtype=complex)
SIZES = [4, 6, 8, 10, 12]
N_CIRCUITS = 5  # random circuits per size


def apply_two_qubit(psi, gate, i, n):
    """Apply a 4x4 gate to qubits (i, i+1) of an n-qubit state vector."""
    psi = np.moveaxis(psi.reshape([2] * n), [i, i + 1], [0, 1]).reshape(4, -1)
    psi = (gate @ psi).reshape([2, 2] + [2] * (n - 2))
    return np.moveaxis(psi, [0, 1], [i, i + 1]).reshape(-1)


def random_two_qubit(rng):
    """A Haar-random 4x4 unitary (QR of a complex Gaussian matrix, phases fixed)."""
    z = (rng.normal(size=(4, 4)) + 1j * rng.normal(size=(4, 4))) / np.sqrt(2)
    q, r = np.linalg.qr(z)
    return q * (np.diag(r) / abs(np.diag(r)))


def brickwork_state(n, depth, rng):
    """|0...0> through `depth` layers of random two-qubit gates on alternating pairs."""
    psi = np.zeros(2**n, dtype=complex); psi[0] = 1
    for layer in range(depth):
        for i in range(layer % 2, n - 1, 2):
            psi = apply_two_qubit(psi, random_two_qubit(rng), i, n)
    return psi


def cnot_ladder_state(n, depth):
    """|0...0>, a Hadamard on qubit 0, then CNOT(k, k+1) for k < depth: one gate per layer."""
    psi = np.zeros(2**n, dtype=complex); psi[0] = 1
    psi = apply_two_qubit(psi, np.kron(H_GATE, np.eye(2)), 0, n)
    for k in range(depth):
        psi = apply_two_qubit(psi, CNOT, k, n)
    return psi


def weight_variance_per_qubit(psi, n):
    """Var(w)/n for the Hamming weight w of a computational-basis measurement."""
    p = np.abs(psi) ** 2
    w = np.array([bin(k).count("1") for k in range(2**n)])
    return float(p @ (w - p @ w) ** 2) / n


curves = {f"CNOT ladder, depth {L}": [weight_variance_per_qubit(cnot_ladder_state(n, L), n) for n in SIZES]
          for L in (1, 2, 3)}
curves["random gates, depth 3"] = [
    np.mean([weight_variance_per_qubit(brickwork_state(n, 3, np.random.default_rng([n, s])), n)
             for s in range(N_CIRCUITS)]) for n in SIZES]
curves["GHZ (ladder of depth n-1)"] = [weight_variance_per_qubit(cnot_ladder_state(n, n - 1), n) for n in SIZES]

print(f"{'Var(w)/n':<28}" + "".join(f"{f'n={n}':>8}" for n in SIZES))
for name, values in curves.items():
    print(f"{name:<28}" + "".join(f"{v:>8.3f}" for v in values))

fig, ax = plt.subplots(figsize=(8.5, 5))
styles = [COLORS["muted"], COLORS["source"], COLORS["quantum"], COLORS["positive"], COLORS["highlight"]]
for (name, values), color in zip(curves.items(), styles):
    ax.plot(SIZES, values, marker="o", color=color, lw=2, label=name)
ax.set_xlabel("number of qubits $n$")
ax.set_ylabel(r"$\mathrm{Var}(w)\,/\,n$")
ax.set_title("Shallow circuits concentrate; the GHZ state does not", pad=10)
ax.legend(frameon=False, fontsize=9.5)
fig.tight_layout()
plt.show()

**Read the figure.** Each curve is $\mathrm{Var}(w)/n$ against the number of qubits, for one family of circuits started at $|0\cdots0\rangle$. The three ladders of fixed depth fall as $n$ grows, exactly as the hand computation said: $(L+1)^2/(4n)$, so $0.083$, $0.187$ and $0.333$ at $n = 12$ for depths 1, 2 and 3. The teal curve, random two-qubit gates of depth 3, stays between $0.15$ and $0.26$: bounded, as the inequality requires, and close to the $0.25$ of a string of independent fair bits. The rose curve is the GHZ state, $n/4$, which reaches $3.0$ at $n = 12$ and keeps climbing: its weight is spread over the whole range from $0$ to $n$. Building it took a ladder of depth $n - 1$, whose light cone covers every qubit. That is the theorem's mechanism on a small scale, checked on these examples: with a fixed light cone, $\mathrm{Var}(w)/n$ stays bounded, and an output whose measured strings spread far apart needs depth that grows with $n$.

## 3. The broader QOT taxonomy — and where this theorem lives

This module committed, deliberately and from `04/03`, to **one** quantum optimal-transport object: the **coupling SDP**, a bipartite density matrix $\rho_{AB} \succeq 0$ with prescribed partial-trace marginals, minimising $\mathrm{tr}(C\,\rho_{AB})$. It is the closest analogue of the Kantorovich linear program, and it carried the entropic regularisation and the information-geometric bridges of the course. But it is *one corner* of a larger landscape — and crucially, **it is not the corner this theorem lives in**. The De Palma et al. limitation runs on a *different* QOT object, the quantum-$W_1$ distance, and seeing why is the point of placing them side by side.

Trevisan's survey is the map. The table situates the families, with the one carrying §2's proof marked.

| Family | Core idea | Relation to this module |
|---|---|---|
| **Coupling SDP** (Golse–Mouhot–Paul 2016; Cole–Eckstein–Friedland–Życzkowski 2023) | bipartite $\rho_{AB} \succeq 0$ with marginals; minimise $\mathrm{tr}(C\,\rho_{AB})$ | **the object this module built** (`04/04`–`04/16`); a static distance *between two states* |
| **Entropic / quantum Sinkhorn** (Peyré–Chizat–Vialard–Solomon 2019 for matrix-valued measures; Feliciangeli–Gerolin–Portinale 2023 for density matrices) | the coupling SDP with a von-Neumann-entropy regulariser; matrix-exp Gibbs kernel | built in `04/06`–`04/08`; the Amari bridge survives the lift |
| **Quantum $W_1$ / Lipschitz** (De Palma–Marvian–Trevisan–Lloyd 2021) | a quantum Lipschitz constant on multi-qubit observables; $W_1$ is its dual | **the tool of §2's proof** — a metric *on many-body states*, with concentration and contraction inequalities; *not* generally equal to the coupling-SDP value |
| **Channel-based** (De Palma–Trevisan 2021) | minimise over the channels that send one state to the other; the matching couplings carry a transposed marginal | survey context; a different object from the coupling SDP |
| **Dynamic / Carlen–Maas** (Carlen–Maas 2014) | replace the continuity equation by a Lindblad-type generator; a $W_2$ on the *dynamics* | survey context; ties to quantum gradient flow |

**These families are not interchangeable.** They do not even all agree on classical (commuting) states: recovering the classical Wasserstein distance there is a property some formulations have and others lack, and for the coupling SDP it depends on the cost, as `04/03` showed. Off the commuting case they answer different questions and return different numbers. The coupling SDP is built to be a *distance between two given states* and reads off a transport cost; the quantum-$W_1$ family is built around a *Lipschitz / observable* structure on many-body systems and comes with the *concentration* and *contraction* machinery that §2 needs. That machinery, Poincaré and transport inequalities that make Lipschitz observables concentrate, is precisely why the limitation theorem is stated in quantum-$W_1$ and not in the coupling SDP. The lesson is the one the course has repeated: there is no single "*the*" quantum optimal transport; the right object is the one whose structure matches the question, and a *limitation-by-concentration* question calls for the Lipschitz / $W_1$ corner of the map.


## 4. Open problems at this frontier

A frontier is honest about what it does not yet know. Three questions sit open right where this module's machinery and this notebook's theorem meet — each a genuine place to start, not a settled matter.

1. **How tight is the depth threshold, and where exactly does advantage survive?** The theorem bounds *shallow, local* circuits; deeper or less-local circuits escape it. The sharp boundary — the precise depth, locality, and cost-Hamiltonian structure at which a provable quantum advantage can still exist — is not fully mapped. The noisy-regime threshold $L = \mathcal{O}(1/p)$ is dramatic but specific to local depolarising noise and problems like Max-Cut; the full phase diagram of *which* problems, depths, and noise models admit advantage is open.

2. **What is the right QOT object for a given limitation, and how do the families relate quantitatively?** The coupling SDP this module built and the quantum-$W_1$ family of §2 agree on commuting states but differ off them, and the *quantitative* relationship between them on general many-body states is not fully characterised. A cleaner dictionary between the QOT families — when a bound in one transfers to another — would let results proved in one corner travel to the others.

3. **Does QOT-based coupling carry over to real, noisy, finite data — the hyperscanning question?** The capstone (`04/11`–`04/16`) measured, on synthetic oscillators, that a richer-embedding quantum measure reads coupling structure the phase-locking value cannot, and that a classical second-harmonic statistic reads the same structure from shorter records than the equal-weight embedding needs. Whether the common geometry of `04/16` earns its place on *real* EEG hyperscanning data, with its noise, nonstationarity, and finite samples, is untested and open. The concentration phenomena of this notebook are a caution worth carrying into that question: high-dimensional quantum objects estimated from limited data can concentrate, and a frontier result about *circuits* is a reminder to ask the same of *estimators*.

These are not the only open problems — but they are the ones this course's machinery, and this notebook's theorem, are best placed to attack.


## Your turn

The warm-up uses the computation of §2; the other two are reasoning prompts.

### Warm-up

Predict, by hand, $\mathrm{Var}(w)/n$ for a CNOT ladder of depth 5 at $n = 12$, then add `5` to the depths of the cell and check. Then say in one or two sentences *why depth is the dial*: what grows with depth, and why the GHZ curve cannot be reached at fixed depth. Finally, the observable $n\,Z_1$ (the first qubit's $Z$, times $n$) is a function of the measured string: what is its Lipschitz constant, and why does the Poincaré inequality say nothing useful about it?

### Go further

A colleague summarises this notebook as "quantum optimal transport proves that VQE does not work." Write the two- or three-sentence correction you would send back. Name the *two conditions* the theorem actually requires (think depth and locality), name the *regime* it targets (near-term / NISQ), and state precisely what it forbids — a class of claims about *shallow* circuits — versus what it leaves entirely open.

### Relate it to the capstone

The capstone (`04/16`) used a QOT measure as an *instrument* pointed at data; this notebook used a QOT distance as a *proof technique* pointed at an algorithm. In a short paragraph, contrast the two uses: in the capstone, the *embedding* was the dial that set what the coupling measure could see; here, the *circuit depth* is the dial that sets how far the output can spread. What is common to both stories about *which* quantum-optimal-transport object was the right tool for the question — the coupling SDP for measuring a fixed pair of states, versus the Wasserstein-1 / Lipschitz objects for bounding what a process can produce?


## What you built

You have crossed from using quantum optimal transport to understanding it as a research instrument with two faces — and you did it without softening either one.

- You made the turn from **QOT-as-measurement to QOT-as-proof-technique**: optimal transport, which read coupling off a density matrix throughout this module, becomes in the hands of De Palma et al. the instrument that *bounds* a computation, through a different object of the family.
- You can state the **De Palma, Marvian, Rouzé & Stilck França (2023)** theorem *accurately*: a limitation on *shallow, local* variational circuits (VQE / QAOA), conditioned on depth and locality and a Lipschitz cost — the output of a circuit with a small light cone obeys a Poincaré inequality, so Lipschitz quantities concentrate as on a product state, which forces QAOA to logarithmic depth on some Max-Cut instances (and, with noise at depth $\mathcal{O}(1/p)$, makes beating classical baselines exponentially unlikely). You can say what it does *not* claim: not that variational methods are useless.
- You placed this module's **coupling SDP inside the broader QOT taxonomy** and saw that the theorem lives in a *different* corner — the quantum-$W_1$ / Lipschitz family, whose concentration and contraction structure is what the proof needs — reinforcing the course's refrain that the right QOT object is the one matched to the question.
- You named three honest **open problems**: the sharp depth/advantage boundary, the quantitative relations among QOT families, and whether QOT coupling survives on real hyperscanning data.

Take the measure of this. You now hold quantum optimal transport as *both* a measurement tool — the coupling measure you demonstrated and bounded in the capstone — *and* a proof technique at the research frontier, a geometry sharp enough to draw a provable line around a family of quantum algorithms. Seeing one mathematics serve both roles, and stating each precisely, is what it means to understand a tool rather than merely run it.

## What's next

One notebook remains. In `20_frontier_synthesis` the whole course comes back into one frame — the greatest hits replayed and the completed classical ↔ quantum dictionary read in full — closing the arc from "moving a pile of mass" to a quantum optimal transport you can both compute with and reason about. Bring this notebook's double vision with you into that reading.


## References

- G. De Palma, M. Marvian, C. Rouzé & D. Stilck França, "Limitations of variational quantum algorithms: a quantum optimal transport approach", *PRX Quantum* **4**, 010309 (2023). DOI:10.1103/PRXQuantum.4.010309 — arXiv:2204.03455. The frontier result of this notebook: transport and concentration inequalities (a quantum (2, ∞)-Poincaré inequality for shallow circuits, transport-cost inequalities for noisy ones) limit what shallow variational quantum algorithms can achieve.
- G. De Palma, M. Marvian, D. Trevisan & S. Lloyd, "The quantum Wasserstein distance of order 1", *IEEE Transactions on Information Theory* **67**, 6627–6643 (2021). DOI:10.1109/TIT.2021.3076442 — arXiv:2009.04469. Defines the quantum-$W_1$ distance via a quantum Lipschitz constant on observables (its dual), and proves the Marton transportation inequality, the Gaussian concentration inequality for Lipschitz observables, and bounds on the contraction coefficients of shallow circuits.
- D. Trevisan, "Quantum optimal transport: an invitation", *Bollettino dell'Unione Matematica Italiana* **18**, 347–360 (2025). DOI:10.1007/s40574-024-00428-5 — the survey whose taxonomy organises §3; situates the coupling SDP, the quantum-$W_1$ / Lipschitz family, the channel-based and dynamic formulations on one map.
- G. De Palma & D. Trevisan, "Quantum optimal transport with quantum channels", *Annales Henri Poincaré* **22**, 3199–3234 (2021). DOI:10.1007/s00023-021-01042-3 — the channel-based formulation of the taxonomy's fourth row.
- F. Golse, C. Mouhot & T. Paul, "On the mean field and classical limits of quantum mechanics", *Communications in Mathematical Physics* **343**, 165–205 (2016). DOI:10.1007/s00220-015-2485-7 — the first transport problem posed over quantum couplings.
- S. Cole, M. Eckstein, S. Friedland & K. Życzkowski, "On quantum optimal transport", *Mathematical Physics, Analysis and Geometry* **26**, 14 (2023). DOI:10.1007/s11040-023-09456-7 — arXiv:2105.06922. The transport problem over all quantum couplings, solved as a semidefinite program, with the SWAP-based cost $\tfrac12(I - \mathrm{SWAP})$; this course uses $I - \mathrm{SWAP}$, twice that cost.
- G. Peyré, L. Chizat, F.-X. Vialard & J. Solomon, "Quantum entropic regularization of matrix-valued optimal transport", *European Journal of Applied Mathematics* **30**, 1079–1102 (2019). DOI:10.1017/S0956792517000274 — entropic regularisation for transport between matrix-valued measures.
- D. Feliciangeli, A. Gerolin & L. Portinale, "A non-commutative entropic optimal transport approach to quantum composite systems at positive temperature", *Journal of Functional Analysis* **285**, 109963 (2023). DOI:10.1016/j.jfa.2023.109963 — entropic optimal transport on density matrices, with the von Neumann entropy as regulariser.

**Previous:** `notebooks/04_QuantumOptimalTransport/18_what_no_classical_data_embedding_can_buy.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/20_frontier_synthesis.ipynb`
